Do decoding on single electrodes; pool results

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "5"  # Limit OpenMP
os.environ["MKL_NUM_THREADS"] = "5"  # Limit MKL (Intel Math Kernel Library)
os.environ["OPENBLAS_NUM_THREADS"] = "5"  # Limit OpenBLAS
os.environ["NUMEXPR_MAX_THREADS"] = "5"  # Limit NumExpr if installed

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from tqdm.auto import tqdm
# from tqdm import tqdm

from src.data import get_electrode_df, add_metadata_features
from src.models.decoding import run_decoding_searchlight_single_electrode

In [ ]:
epochs_path = "outputs/epochs_preprocessed"
outdir = "tmp_single_electrode"
save_outcomes = False
smoke_test = False

filter_speech_responsive = True
prediction_target = "behavior_categorical"

window_size = 30
stride = 10

randomize = False

In [ ]:
all_epoch_paths = list(Path(epochs_path).glob("*.fif"))

In [ ]:
epochs = {}
for path in tqdm(all_epoch_paths):
    subject_name = re.findall("(EC[\d]+)_epo", str(path))[0]
    epochs[subject_name] = mne.read_epochs(str(path))
    epochs[subject_name].metadata = add_metadata_features(epochs[subject_name].metadata)

In [ ]:
electrode_df = pd.concat([get_electrode_df(subject_name) for subject_name in epochs.keys()],
                         keys=epochs.keys(), names=["subject"])
electrode_df["roi"] = electrode_df.roi.astype(str)
electrode_df = electrode_df.droplevel("electrode_name")

# Drop electrodes metadata which don't have corresponding data
for subject, epochs_i in epochs.items():
    electrode_df.loc[subject, "keep"] = np.arange(len(electrode_df.loc[subject])) < len(epochs_i.info["ch_names"])
electrode_df = electrode_df[electrode_df["keep"]].drop(columns="keep")

electrode_df

## Find speech-responsive electrodes

In [ ]:
# power threshold relative to pre-speech baseline which defines a "speech responsive" electrode
# if we see absolute value change >= this threshold, call the electrode speech-responsive
speech_responsive_threshold = 0.3

In [ ]:
# demo this
dd = next(iter(epochs.values())).copy().apply_baseline((-0.1, 0)).average().crop(tmin=0, tmax=0.9).get_data()
keep = np.abs(dd).max(axis=1) > speech_responsive_threshold

f, ax = plt.subplots(figsize=(8, 4))
for line, k in zip(dd, keep):
    plt.plot(line, color="r" if k else "k", alpha=0.1)

In [ ]:
for subject, epochs_i in epochs.items():
    epochs_i = epochs_i.copy().apply_baseline((-0.1, 0)).average().crop(tmin=0, tmax=0.9).get_data()
    assert epochs_i.ndim == 2
    speech_responsive_i = np.abs(epochs_i).max(axis=1) > speech_responsive_threshold

    if len(speech_responsive_i) > len(electrode_df.loc[subject]):
        speech_responsive_i = speech_responsive_i[:len(electrode_df.loc[subject])]
    electrode_df.loc[subject, "speech_responsive"] = speech_responsive_i

In [ ]:
electrode_df = electrode_df.astype({"speech_responsive": bool})

## Decoding

In [ ]:
train_scores, test_scores, outcomes, models = \
    run_decoding_searchlight_single_electrode(
        epochs=epochs,
        electrode_df=electrode_df,
        filter_speech_responsive=filter_speech_responsive,
        target=prediction_target,
        window_size=window_size,
        stride=stride,
        smoke_test=smoke_test,
        randomize=randomize,
        return_outcomes=save_outcomes,
        strategy="train-test",
    )

In [ ]:
train_scores_df = pd.concat(
    {key: pd.DataFrame(scores_i) for key, scores_i in train_scores.items()},
    names=["subject", "electrode_idx", "phoneme_pair", "smin", "smax", "fold"])
train_scores_df["target"] = prediction_target
train_scores_df

In [ ]:
scores_df = pd.concat(
    {key: pd.DataFrame(scores_i) for key, scores_i in test_scores.items()},
    names=["subject", "electrode_idx", "phoneme_pair", "smin", "smax", "fold"])
scores_df["target"] = prediction_target
scores_df

In [ ]:
def get_fit_C(model):
    if hasattr(model, "C"):
        return model.C
    elif hasattr(model, "C_"):
        ret = model.C_
        if isinstance(ret, np.ndarray):
            return ret[0]
        else:
            return ret
    else:
        raise ValueError("unknown model type")


hparam_df = pd.DataFrame(
    [{"subject": subject, "electrode_idx": electrode_idx,
      "phoneme_pair": phoneme_pair,
      "smin": smin, "smax": smax,
      "target": prediction_target,
      "fold": j,
      "C": get_fit_C(fold_fit.steps[-1][1])}
     for (subject, electrode_idx, phoneme_pair, smin, smax), fitted_i in models.items()
     for j, fold_fit in enumerate(fitted_i)])
hparam_df = hparam_df.set_index(["target", "subject", "electrode_idx", "phoneme_pair", "smin", "smax", "fold"])

In [ ]:
merged_df = pd.merge(scores_df, hparam_df,
                     left_index=True, right_index=True,
                     how="left", validate="1:1")
merged_df["log_C"] = np.log10(merged_df["C"])

In [ ]:
sns.histplot(data=merged_df, x="log_C", multiple="dodge")

In [ ]:
train_scores_df.to_csv(Path(outdir) / "train_scores.csv")
scores_df.to_csv(Path(outdir) / "scores.csv")
hparam_df.to_csv(Path(outdir) / "hparams.csv")

In [ ]:
torch.save({
    "models": models,
    "outcomes": outcomes
}, Path(outdir) / "outcomes.pt")